In [1]:
import sys
import os
from pathlib import Path
# Ensure the repository root is on sys.path so imports work when running the notebook
repo_root = Path('..').resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print('Added to sys.path:', repo_root)

Added to sys.path: C:\Users\p00102977\Desktop\Projects\API\FInal Repo content\drl-cognito-ai


In [2]:
from medical_potential.clinical_efficacy.trial_fetcher import _normalise_phase, _normalise_date, _normalise_location, _normalise_row

[OK]  Gemini Model initialised (key: ...Y-Yz9Q) | max concurrent calls: 5


## Normalize the table data

In [3]:
import pandas as pd

clinical_efficacy_data = pd.read_csv(
    r"input/clinical_efficacy/clinical_efficacy_glp1_before_normalizing_columns.csv",
    keep_default_na=False,
)

In [4]:
clinical_efficacy_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2692 entries, 0 to 2691
Data columns (total 41 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   molecule_name                 2692 non-null   object 
 1   registry_source               2692 non-null   object 
 2   trial_id                      2692 non-null   object 
 3   acronym                       2692 non-null   object 
 4   dosage                        2692 non-null   object 
 5   phase                         2692 non-null   object 
 6   trial_title                   2692 non-null   object 
 7   trial_study                   2692 non-null   object 
 8   trial_size                    2692 non-null   object 
 9   trial_location                2692 non-null   object 
 10  secondary_locations           2692 non-null   object 
 11  trial_start_date              2692 non-null   object 
 12  trial_completion_date         2692 non-null   object 
 13  pha

In [5]:
clinical_efficacy_data

,molecule_name,registry_source,trial_id,acronym,dosage,phase,trial_title,trial_study,trial_size,trial_location,...,source_url,llm_confidence,llm_confidence_rationale,trial_source,efficacy_score,efficacy_data_coverage,efficacy_score_breakdown,efficacy_narrative_rationale,created_at,updated_at
0,Na-931,ClinicalTrials.gov,NCT06732245,,150 mg QD,PHASE2,"A Randomized, Double-Blind, Placebo-Controlled...",INTERVENTIONAL | RANDOMIZED; FACTORIAL; TREATM...,224,United States,...,https://clinicaltrials.gov/study/NCT06732245,1.0,Auto-assigned: trial sourced directly from Cli...,ctgov_trials,1.2,"1/4 endpoints scored (missing: hba1c, mash, alt)",weight_loss | adj=12.58% score=3 weight=4...,Na-931 achieved an overall clinical efficacy s...,2026-08-20 15:34:25.756356 UTC,
1,Na-931,ClinicalTrials.gov,NCT06615700,T2DM,150 mg QD,PHASE1,"A Randomized, Double-Blind, Placebo-Controlled...",INTERVENTIONAL | RANDOMIZED; PARALLEL; TREATME...,74,Australia,...,https://clinicaltrials.gov/study/NCT06615700,1.0,Auto-assigned: trial sourced directly from Cli...,ctgov_trials,1.2,"1/4 endpoints scored (missing: hba1c, mash, alt)",weight_loss | adj=12.58% score=3 weight=4...,Na-931 achieved an overall clinical efficacy s...,2026-08-20 15:34:25.756356 UTC,
2,Na-931,ClinicalTrials.gov,NCT06564753,,150 mg QD,PHASE2,"A Phase 2 Randomized, Double-Blind, Placebo-Co...",INTERVENTIONAL | RANDOMIZED; PARALLEL; TREATME...,126,Australia,...,https://clinicaltrials.gov/study/NCT06564753,1.0,Auto-assigned: trial sourced directly from Cli...,ctgov_trials,1.2,"1/4 endpoints scored (missing: hba1c, mash, alt)",weight_loss | adj=12.58% score=3 weight=4...,Na-931 achieved an overall clinical efficacy s...,2026-08-20 15:34:25.756356 UTC,
3,Na-931,GD Clinical Trials,GDC30034841,,150 mg QD,Phase II/III,A Phase IIb/III Study of NA-931 for the treatm...,Treatment,0,,...,,1.0,The trial title explicitly mentions NA-931 and...,ctgov_port,1.2,"1/4 endpoints scored (missing: hba1c, mash, alt)",weight_loss | adj=12.58% score=3 weight=4...,Na-931 achieved an overall clinical efficacy s...,2026-08-20 15:34:25.756356 UTC,
4,Na-931,GD Clinical Trials,GDC20047745,,120 mg QD,Phase III,A Phase III Clinical Protocol Evaluating the S...,Double Blind; Multi-centered; Placebo-controll...,2,,...,,1.0,The trial title explicitly names NA-931 and th...,ctgov_port,1.2,"1/4 endpoints scored (missing: hba1c, mash, alt)",weight_loss | adj=12.58% score=3 weight=4...,Na-931 achieved an overall clinical efficacy s...,2026-08-20 15:34:25.756356 UTC,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2687,Visepegenatide,ChiCTR,ChiCTR2100049282,N/A,150 μg OW,Phase 1,A Phase I study to evaluate the effects of PB-...,Drug-Drug Interaction,,China,...,https://www.chictr.org.cn/showproj.aspx?proj=1...,1.0,The trial title clearly names PB-119 in a Phas...,fetch_trial_ids,0.8,"2/4 endpoints scored (missing: mash, alt)",weight_loss | adj=1.81% score=1 weight=40...,Visepegenatide achieved a clinical efficacy sc...,2026-08-20 15:43:28.084222 UTC,
2688,Visepegenatide,ChiCTR,ChiCTR2000034606,N/A,150 μg OW,Phase 1,A study to evaluate the impact of PB-119 injec...,Drug-Drug Interaction,,China,...,https://www.chictr.org.cn/showproj.aspx?proj=5...,1.0,The trial explicitly studies the impact of PB-...,fetch_trial_ids,0.8,"2/4 endpoints scored (missing: mash, alt)",weight_loss | adj=1.81% score=1 weight=40...,Visepegenatide achieved a clinical efficacy sc...,2026-08-20 15:43:28.084222 UTC,
2689,Visepegenatide,Innovator Website,CTR20150321,,50 μg OW,Phase 1,Phase I Clinical Trial of PB-119 Injection: A ...,Interventional | Pharmacokinetics and Pharmaco...,36,China,...,http://www.chinadrugtrials.org.cn/,1.0,"The trial title explicitly names PB-119, a kno...",innovator_web,0.8,"2/4 endpoints scored (missing: mash, alt)",weight_loss | adj=1.81% score=1 weight=40...,Visepegenatide achieved a clinical efficacy sc...,2026-08-20 15:43:28.084222 UTC,
2690,Visepegenatide,Innovator Website,CTR20201492,,1

In [6]:
import pandas as pd

schema_columns = [
    "molecule_name", "registry_source", "trial_id", "acronym", "dosage",
    "phase", "trial_title", "trial_study", "trial_size", "trial_location",
    "secondary_locations", "trial_start_date", "trial_completion_date",
    "phase_status", "hba1c_change_pct", "hba1c_duration", "hba1c_rationale",
    "hba1c_confidence", "weight_change_pct", "weight_duration", "weight_rationale",
    "weight_confidence", "alt_reduction_pct", "alt_duration", "alt_rationale",
    "alt_confidence", "mash_change_pct", "mash_duration", "mash_rationale",
    "mash_confidence", "company_name", "source_url", "llm_confidence",
    "llm_confidence_rationale", "trial_source", "efficacy_score",
    "efficacy_data_coverage", "efficacy_score_breakdown", "efficacy_narrative_rationale",
    "created_at", "updated_at",
]

float_columns = [
    "hba1c_confidence", "weight_confidence", "alt_confidence", "mash_confidence",
    "llm_confidence", "efficacy_score",
]
timestamp_columns = ["created_at", "updated_at"]
string_columns = [
    column for column in schema_columns
    if column not in float_columns and column not in timestamp_columns
]


def impose_clinical_efficacy_schema(dataframe):
    """Return a dataframe with the clinical efficacy table's columns and dtypes."""
    result = dataframe.reindex(columns=schema_columns).copy()

    for column in string_columns:
        result[column] = result[column].astype("string")

    for column in float_columns:
        result[column] = pd.to_numeric(result[column], errors="coerce").astype("Float64")

    for column in timestamp_columns:
        result[column] = pd.to_datetime(result[column], errors="coerce", utc=True)

    return result


# Stage 1: impose the table schema on the imported data.
clinical_efficacy_data = impose_clinical_efficacy_schema(clinical_efficacy_data)

# Stage 2: normalize each schema-shaped row.
normalized_rows = []
for source_row in clinical_efficacy_data.to_dict(orient="records"):
    # normalized_row = {
    #     key: ("" if pd.isna(value) else value)
    #     for key, value in source_row.items()
    # }
    _normalise_row(source_row)
    normalized_rows.append(source_row)

# Stage 3: impose the table schema again on normalized output.
normalized_clinical_efficacy_data = impose_clinical_efficacy_schema(
    pd.DataFrame(normalized_rows)
)

# Preserve authoritative confidence for rows sourced from GD Clinical Trials.
gd_trials_mask = normalized_clinical_efficacy_data["registry_source"].eq("GD Clinical Trials")
normalized_clinical_efficacy_data.loc[gd_trials_mask, "llm_confidence"] = 1.0
normalized_clinical_efficacy_data.loc[
    gd_trials_mask, "llm_confidence_rationale"
] = "Auto-assigned: trial sourced directly from GD Clinical Trials (authoritative primary registry)."

# Save the schema-aligned, normalized dataframe as Parquet.
normalized_clinical_efficacy_data.to_parquet(
    "normalized_clinical_efficacy_data.parquet",
    index=False,
)

normalized_clinical_efficacy_data

,molecule_name,registry_source,trial_id,acronym,dosage,phase,trial_title,trial_study,trial_size,trial_location,...,source_url,llm_confidence,llm_confidence_rationale,trial_source,efficacy_score,efficacy_data_coverage,efficacy_score_breakdown,efficacy_narrative_rationale,created_at,updated_at
0,Na-931,ClinicalTrials.gov,NCT06732245,,150 mg QD,Phase 2,"A Randomized, Double-Blind, Placebo-Controlled...",INTERVENTIONAL | RANDOMIZED; FACTORIAL; TREATM...,224,United States,...,https://clinicaltrials.gov/study/NCT06732245,1.0,Auto-assigned: trial sourced directly from Cli...,ctgov_trials,1.2,"1/4 endpoints scored (missing: hba1c, mash, alt)",weight_loss | adj=12.58% score=3 weight=4...,Na-931 achieved an overall clinical efficacy s...,2026-08-20 15:34:25.756356+00:00,NaT
1,Na-931,ClinicalTrials.gov,NCT06615700,T2DM,150 mg QD,Phase 1,"A Randomized, Double-Blind, Placebo-Controlled...",INTERVENTIONAL | RANDOMIZED; PARALLEL; TREATME...,74,Australia,...,https://clinicaltrials.gov/study/NCT06615700,1.0,Auto-assigned: trial sourced directly from Cli...,ctgov_trials,1.2,"1/4 endpoints scored (missing: hba1c, mash, alt)",weight_loss | adj=12.58% score=3 weight=4...,Na-931 achieved an overall clinical efficacy s...,2026-08-20 15:34:25.756356+00:00,NaT
2,Na-931,ClinicalTrials.gov,NCT06564753,,150 mg QD,Phase 2,"A Phase 2 Randomized, Double-Blind, Placebo-Co...",INTERVENTIONAL | RANDOMIZED; PARALLEL; TREATME...,126,Australia,...,https://clinicaltrials.gov/study/NCT06564753,1.0,Auto-assigned: trial sourced directly from Cli...,ctgov_trials,1.2,"1/4 endpoints scored (missing: hba1c, mash, alt)",weight_loss | adj=12.58% score=3 weight=4...,Na-931 achieved an overall clinical efficacy s...,2026-08-20 15:34:25.756356+00:00,NaT
3,Na-931,GD Clinical Trials,GDC30034841,,150 mg QD,Phase 2/Phase 3,A Phase IIb/III Study of NA-931 for the treatm...,Treatment,N/A,,...,,1.0,Auto-assigned: trial sourced directly from GD ...,ctgov_port,1.2,"1/4 endpoints scored (missing: hba1c, mash, alt)",weight_loss | adj=12.58% score=3 weight=4...,Na-931 achieved an overall clinical efficacy s...,2026-08-20 15:34:25.756356+00:00,NaT
4,Na-931,GD Clinical Trials,GDC20047745,,120 mg QD,Phase 3,A Phase III Clinical Protocol Evaluating the S...,Double Blind; Multi-centered; Placebo-controll...,2,,...,,1.0,Auto-assigned: trial sourced directly from GD ...,ctgov_port,1.2,"1/4 endpoints scored (missing: hba1c, mash, alt)",weight_loss | adj=12.58% score=3 weight=4...,Na-931 achieved an overall clinical efficacy s...,2026-08-20 15:34:25.756356+00:00,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2687,Visepegenatide,ChiCTR,ChiCTR2100049282,N/A,150 μg OW,Phase 1,A Phase I study to evaluate the effects of PB-...,Drug-Drug Interaction,N/A,China,...,https://www.chictr.org.cn/showproj.aspx?proj=1...,1.0,The trial title clearly names PB-119 in a Phas...,fetch_trial_ids,0.8,"2/4 endpoints scored (missing: mash, alt)",weight_loss | adj=1.81% score=1 weight=40...,Visepegenatide achieved a clinical efficacy sc...,2026-08-20 15:43:28.084222+00:00,NaT
2688,Visepegenatide,ChiCTR,ChiCTR2000034606,N/A,150 μg OW,Phase 1,A study to evaluate the impact of PB-119 injec...,Drug-Drug Interaction,N/A,China,...,https://www.chictr.org.cn/showproj.aspx?proj=5...,1.0,The trial explicitly studies the impact of PB-...,fetch_trial_ids,0.8,"2/4 endpoints scored (missing: mash, alt)",weight_loss | adj=1.81% score=1 weight=40...,Visepegenatide achieved a clinical efficacy sc...,2026-08-20 15:43:28.084222+00:00,NaT
2689,Visepegenatide,Innovator Website,CTR20150321,,50 μg OW,Phase 1,Phase I Clinical Trial of PB-119 Injection: A ...,Interventional | Pharmacokinetics and Pharmaco...,36,China,...,http://www.chinadrugtrials.org.cn/,1.0,"The trial title explicitly names PB-119, a kno...",innovator_web,0.8,"2/4 endpoints scored (missing: mash, alt)",weight_loss | adj=1.81% score=1 weight=40...,Visepegenatide achieved a clinical efficacy sc...,2026-08-20 15:43:28.084222+00:00,NaT


In [ ]:
normalized_clinical_efficacy_data.to_csv('normalized_clinical_efficacy_data.csv', index = False)

In [ ]:
phase_list = ['PHASE1','Phase III','Phase I','Technology Appraisal','Phase 3','Phase 1','Phase N/A','Phase 4','Phase 0','Phase 1/2','Phase 3b','PHASE3','PHASE2','Phase 2','Therapeutic confirmatory  (Phase III)','PHASE4','Phase II','Human Pharmacology (Phase I)- Bioequivalence Study','Human Pharmacology (Phase I)-  Other','Therapeutic exploratory (Phase II)','Phase II and Phase III (Integrated)','NA','EARLY_PHASE1','PHASE1; PHASE2','PHASE2; PHASE3','2','3','Filed','Phase 3a','Phase III and phase IV (Integrated)','Phase I/II','Phase IV','Phase 1b','Phase 1a',]

In [ ]:
cleaned_phase_list = [_normalise_phase(phase_item) for phase_item in phase_list]

In [ ]:
list(zip(phase_list, cleaned_phase_list))

## Checking date normalization

In [ ]:
date_list = ['(01-03-2031)','(01-02-2027)','(01-08-2031)','(01-09-2027)','2027-01','2028-05','2027-08','2027-03','2027-09','2027-05','2030-03','2031-01','2027-10','2031-08','2028-06','2028-03','2029-11','(05-06-2029)','(27-12-2031)','(04-09-2026)','(22-12-2026)','2014-11','2013-05','2016-07','2017-05','2011-07','2016-11','2021-06','2013-11','2013-10','2013-03','2022-09','2012-02','2010-06','2010-02','2010-07','2009-09','2009-12','2012-06','2017-01','2025-10','2014-07','2008-05','2016-02','2014-04','2015-12','2007-11','2013-04','2015-04','2013-12','2009-08','2008-03','2018-09','2007-03','2007-10','2029-06','2015-08','2011-05','2012-01','2010-10','2009-03','2006-05','2016-04','2009-04','(28-04-2028)','(01-03-2027)','(30-11-2026)','(30-09-2026)','(31-10-2027)','(07-05-2027)','(31-12-2027)','(24-09-2026)','(31-12-2026)','(30-04-2028)','2028-02','2028-12','2029-07','(01-09-2026)','(28-02-2027)','28-02-2027']

In [ ]:
cleaned_date_list = [_normalise_date(date_item) for date_item in date_list]

In [ ]:
list(zip(date_list, cleaned_date_list))